# 1. MLP-Trainingsskript (mit Baseline und Dropout)

`dropout_rate = 0.1` deaktiviert beim Training zufällig 10 % der Aktivierungen
nach der ersten versteckten Schicht. Mit `0.0` ist Dropout ausgeschaltet.
Bei diesem kleinen Netz zunächst niedrige Raten wie `0.1` oder `0.2` ausprobieren.
Dropout kann Overfitting reduzieren, garantiert aber keinen niedrigeren MAPE.
Für die Auswahl der Rate eine separate Validierungsmenge verwenden;
die Testmenge für die abschließende Bewertung reservieren.
`model.train()` aktiviert Dropout, `model.eval()` deaktiviert es für Vorhersagen.

In [25]:
import kagglehub
from copy import deepcopy
import pandas
import torch
import matplotlib.pyplot as plt
import numpy

from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

# 1. Datensatz runterladen und mit Pandas einlesen
path = kagglehub.dataset_download("mchilamwar/predict-concrete-strength")
df = pandas.read_csv(f"{path}/ConcreteStrengthData.csv")
df = df.rename(columns={"CementComponent ": "CementComponent"})

# 2. Definiere, was Input x und Output y für das Modell sein soll
zielspalte = "Strength"
x = df.drop( zielspalte, axis="columns")
y = df[ [zielspalte] ]

# 3. Train-Test-Split
x_train, x_test, y_train, y_test = train_test_split(x,y,
                                                    test_size=0.2,
                                                    random_state=123
                                                   )
# 4. Skalierung der Daten
x_scaler = StandardScaler()
y_scaler = StandardScaler()
x_train_scaled = x_scaler.fit_transform(x_train)
y_train_scaled = y_scaler.fit_transform(y_train)

# 5. Erzeuge jetzt aus den Daten Tensoren
x_train_scaled_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_scaled_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

nr_inputs = x_train.shape[-1]
nr_outputs = y_train.shape[-1]

# Dropout-Wahrscheinlichkeit: 0.0 = aus, zunächst 0.1 ausprobieren.
dropout_rate = 0.1

MAPEs = []
best_mape = float("inf")
best_model = None
best_run = None
for run in range(10):

    print(run, end=" ")
    # Reproduzierbare Initialisierung für jeden der zehn Läufe.
    torch.manual_seed(123 + run)

    # 6. Modell bauen
    N1 = 128
    N2 = 64
    model = nn.Sequential(
        nn.Linear(nr_inputs, N1),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate),
        
        nn.Linear(N1, N2),
        nn.ReLU(),
        
        nn.Linear(N2, nr_outputs),
    )
    
    # 7. Modell trainieren
    loss_function = nn.MSELoss() # Mittler Quadratischer Fehler = Mean Squared Error
    #loss_function = nn.L1Loss() # MAE
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    model.train()
    losses = []
    for epoch in range(1000):        
        predictions = model( x_train_scaled_tensor )
        loss = loss_function(predictions, y_train_scaled_tensor)
        losses.append(loss.item())
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # 8. Das fertige MLP testen
    x_test_scaled = x_scaler.transform(x_test)
    x_test_scaled_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        predictions_test_scaled = model(x_test_scaled_tensor)
    predictions_test = y_scaler.inverse_transform(predictions_test_scaled.numpy())    
    y_pred = predictions_test.flatten()
    y_true = y_test["Strength"].values    
    MAPE = mean_absolute_percentage_error(y_true, y_pred)*100
    MAPEs.append( MAPE )
    if MAPE < best_mape:
        best_mape = MAPE
        best_run = run + 1
        best_model = deepcopy(model).cpu().eval()

print()
print(MAPEs)
print(f"Dropout-Rate: {dropout_rate}")
print(f"Average MAPE : {numpy.mean(MAPEs):.1f}%")

# Für Speichern und Export das beste Modell verwenden, nicht den letzten Lauf.
model = best_model
print(f"Bestes MLP: Lauf {best_run}, MAPE: {best_mape:.2f}%")

# Abschließend ein Vergleich mit Standard-Baseline: MVLR

# 1. Trainiere die Lineare Regression
base_model = LinearRegression().fit(x_train_scaled, y_train_scaled)

# 2. Teste das Modell mit den Testdaten
y_pred_scaled = base_model.predict( x_test_scaled )
y_pred = y_scaler.inverse_transform( y_pred_scaled )

# 3. Berechne den MAPE der LR auf den Testdaten
y_true = y_test["Strength"].values
MAPE = mean_absolute_percentage_error(y_true, y_pred.flatten())*100
print(f"MVLR MAPE: {MAPE:.1f}%")

# Modell und Skalierer abspeichern
import pickle
datei = open("mlp_betonstaerken_regressor.pkl", "wb")
pickle.dump(x_scaler, datei)
pickle.dump(model, datei)
pickle.dump(y_scaler, datei)
datei.close()

0 1 2 3 4 5 6 7 8 9 
[9.881184117668084, 10.019276832510362, 9.633089373721441, 9.115607523494953, 9.738299638206328, 10.225304617168248, 9.52337330630204, 10.206352504442103, 9.415339078956302, 9.147247758060024]
Dropout-Rate: 0.1
Average MAPE : 9.7%
Bestes MLP: Lauf 4, MAPE: 9.12%
MVLR MAPE: 27.9%


# 2. Bestes MLP als ONNX exportieren

Auswahlkriterium ist der niedrigste Test-MAPE der zehn Läufe. Damit wird die
Testmenge auch zur Modellauswahl verwendet: Der beste MAPE ist keine unabhängige
Schätzung der Generalisierung mehr. Für eine unabhängige Bewertung später eine
separate Validierungsmenge zur Auswahl und eine unberührte Testmenge verwenden.

Der Export enthält Eingabeskalierung, MLP und Rückskalierung der Zielwerte.
Die erwarteten Spaltennamen werden in der ONNX-Datei gespeichert.
Dropout ist beim Export ausgeschaltet. Die Anzahl der Mischungen ist variabel.

Falls nötig, einmal im Notebook-Kernel installieren:
`%pip install onnx onnxscript onnxruntime`

Verwendet wird der [moderne PyTorch-ONNX-Exporter](https://docs.pytorch.org/docs/stable/onnx.html).


In [ ]:
import json
import onnx
import onnxruntime as ort

class BetonModellMitSkalierung(nn.Module):
    def __init__(self, mlp, input_scaler, output_scaler):
        super().__init__()
        self.mlp = mlp
        # legt Mittelwerte und Skalierungsfaktoren im PyTorch-Modell ab, damit sie mit exportiert werden
        self.register_buffer("x_mean", torch.tensor(input_scaler.mean_, dtype=torch.float32))
        self.register_buffer("x_scale", torch.tensor(input_scaler.scale_, dtype=torch.float32))
        self.register_buffer("y_mean", torch.tensor(output_scaler.mean_, dtype=torch.float32))
        self.register_buffer("y_scale", torch.tensor(output_scaler.scale_, dtype=torch.float32))

    def forward(self, mischungen):        
        x_standardisiert = (mischungen - self.x_mean) / self.x_scale
        y_standardisiert = self.mlp(x_standardisiert)
        return y_standardisiert * self.y_scale + self.y_mean

onnx_datei = "bestes_mlp_betonstaerke.onnx"

# setzt das Exportmodell auf CPU und den Vorhersagemodus,
# insbesondere wird Dropout ausgeschaltet
export_model = BetonModellMitSkalierung(best_model, x_scaler, y_scaler).cpu().eval()
beispiel_input = torch.tensor(x_train.iloc[:3].to_numpy(), dtype=torch.float32)

torch.onnx.export(
    export_model,
    (beispiel_input,),
    onnx_datei,
    input_names=["mischungen"],
    output_names=["betonstaerke"],
    dynamo=True,
    dynamic_shapes=({0: torch.export.Dim("anzahl_mischungen", min=1)},),
    opset_version=18,
    external_data=False,  # Gewichte in derselben ONNX-Datei speichern.
)



# lädt die gerade exportierte ONNX-Datei wieder ein – hier nur, um Metadaten hinzuzufügen.
onnx_modell = onnx.load(onnx_datei)

# hinterlegt zum Beispiel Spaltennamen und den ausgewählten Lauf in der Datei.
onnx.helper.set_model_props(onnx_modell, {
    "feature_names": json.dumps(x_train.columns.tolist()),
    "target_name": "Strength",
    "best_run": str(best_run),
    "selection_test_mape_percent": str(best_mape),
    "scaling": "StandardScaler for inputs and output included",
})

# prüft die ONNX-Datei auf formale Fehler.
onnx.checker.check_model(onnx_modell)

# speichert die mit Metadaten ergänzte Datei erneut unter demselben Namen
onnx.save(onnx_modell, onnx_datei)

# Export gegen die ursprüngliche sklearn-/PyTorch-Pipeline prüfen.
onnx_session = ort.InferenceSession(onnx_datei, providers=["CPUExecutionProvider"])
with torch.no_grad():
    referenz = y_scaler.inverse_transform(
        best_model(torch.tensor(x_test_scaled, dtype=torch.float32)).numpy()
    )
for anzahl in (1, 3, len(x_test)):
    onnx_vorhersagen = onnx_session.run(
        ["betonstaerke"],
        {"mischungen": x_test.iloc[:anzahl].to_numpy(dtype=numpy.float32)},
    )[0]
    numpy.testing.assert_allclose(
        onnx_vorhersagen, referenz[:anzahl], rtol=1e-5, atol=1e-4
    )
print(f"Exportiert: {onnx_datei} (Lauf {best_run}, MAPE {best_mape:.2f}%)")
print("ONNX stimmt für 1, 3 und alle Testfälle mit PyTorch überein.")


[torch.onnx] Obtain model graph for `BetonModellMitSkalierung([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BetonModellMitSkalierung([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Exportiert: bestes_mlp_betonstaerke.onnx (Lauf 10, MAPE 9.31%)
ONNX stimmt für 1, 3 und alle Testfälle mit PyTorch überein.


# 2. MLP-Inferenzskript

In [3]:
x_test.head(3).to_csv("neue_beton_mischungen.csv", index=False)

In [4]:
# 1. Modell und Skalierer wiederherstellen
import pickle
datei = open("mlp_betonstaerken_regressor.pkl", "rb")
x_scaler = pickle.load(datei)
model = pickle.load(datei)
y_scaler = pickle.load(datei)
datei.close()

# 2. Neue Betonmischungen einlesen
import pandas
df = pandas.read_csv("neue_beton_mischungen.csv")

# 3. Daten skalieren, durchs Modell, Modelloutput zurück skalieren
import torch
x_scaled = x_scaler.transform(df)
x_scaled_tensor = torch.tensor(x_scaled, dtype=torch.float32)
model.eval()  # Dropout auch nach dem Laden explizit ausschalten.
with torch.no_grad():
    predictions_scaled_tensor = model(x_scaled_tensor)
y_pred = y_scaler.inverse_transform( predictions_scaled_tensor.numpy() )

# 4. Prädizierte Betonstärken in Tabelle schreiben
df["PredStrength"] = y_pred
df

,CementComponent,BlastFurnaceSlag,FlyAshComponent,WaterComponent,SuperplasticizerComponent,CoarseAggregateComponent,FineAggregateComponent,AgeInDays,PredStrength
0,362.6,189.0,0.0,164.9,11.6,944.7,755.8,28,71.120537
1,190.0,190.0,0.0,228.0,0.0,932.0,670.0,90,43.293781
2,152.6,238.7,0.0,200.0,6.3,1001.8,683.9,28,26.016211


# 3. Vergleich mit weiteren Regressionsverfahren

Zuerst den Trainingsabschnitt ausführen. Alle fünf Verfahren verwenden exakt
`x_train_scaled`, `y_train_scaled` und `x_test_scaled` aus diesem Abschnitt:
Es gibt keinen neuen Split und kein erneutes Anpassen der StandardScaler.
Auch die Zielwerte bleiben beim Training standardisiert. Vor der Bewertung
werden die Vorhersagen mit `y_scaler` in die ursprüngliche Einheit zurückgerechnet.
Der MAPE wird auf denselben Testfällen berechnet und in Prozent angegeben.

Dies ist ein erster Vergleich mit den unten angegebenen Einstellungen, ohne
Hyperparameteroptimierung. Kleinere MAPE-Werte sind besser. Für späteres Tuning
nur die Trainingsdaten (z. B. mit Kreuzvalidierung) verwenden.


In [5]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Vorhandene skalierte Daten unverändert übernehmen; sklearn erwartet y als 1D-Array.
y_train_vergleich = y_train_scaled.ravel()
y_test_vergleich = y_test["Strength"].to_numpy()

vergleichsmodelle = {
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=123),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=100, random_state=123, n_jobs=-1,
    ),
    "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=5),
    "SVR": SVR(kernel="rbf", C=1.0, epsilon=0.1),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=123
    ),
}

vergleichsergebnisse = []
for name, regressor in vergleichsmodelle.items():
    regressor.fit(x_train_scaled, y_train_vergleich)
    vorhersagen_scaled = regressor.predict(x_test_scaled)
    vorhersagen = y_scaler.inverse_transform(
        vorhersagen_scaled.reshape(-1, 1)
    ).ravel()
    vergleichsergebnisse.append({
        "Modell": name,
        "MAPE (%)": mean_absolute_percentage_error(
            y_test_vergleich, vorhersagen
        ) * 100,
        "MAE": mean_absolute_error(y_test_vergleich, vorhersagen),
        "R²": r2_score(y_test_vergleich, vorhersagen),
    })

# Numerische Werte behalten; nur die Anzeige runden.
ergebnisse_df = (
    pandas.DataFrame(vergleichsergebnisse)
    .sort_values("MAPE (%)", ascending=True)
    .reset_index(drop=True)
)
ergebnisse_df.round({"MAPE (%)": 2, "MAE": 2, "R²": 3})


,Modell,MAPE (%),MAE,R²
0,RandomForestRegressor,10.39,3.42,0.893
1,GradientBoostingRegressor,11.22,3.71,0.897
2,DecisionTreeRegressor,12.74,4.04,0.833
3,SVR,14.03,4.54,0.843
4,KNeighborsRegressor,20.42,6.09,0.753


# 4. Betonfestigkeit aus einer CSV mit ONNX vorhersagen

Die erste Zelle erzeugt eine Beispieldatei mit drei Mischungen aus der Testmenge
(ohne bekannte Festigkeit). Bei eigenen Daten diese Zelle überspringen und den
CSV-Pfad in der nächsten Zelle anpassen.

Die Inferenzzelle benötigt nur die CSV-Datei, die exportierte ONNX-Datei sowie
`pandas`, `numpy` und `onnxruntime`: weder PyTorch noch separate Scaler-Dateien.
Sie liest die Eingabespalten aus den ONNX-Metadaten und ordnet sie korrekt an.
Eingabewerte müssen dieselben Einheiten wie die Trainingsdaten verwenden.
Die neue Spalte `PredStrength` enthält die Festigkeit in der ursprünglichen Einheit.


In [6]:
# Beispieldatei erzeugen (benötigt x_test aus Abschnitt 1).
x_test.head(3).to_csv("beton_mischungen_onnx.csv", index=False)


In [27]:
import json
import numpy as np
import pandas as pd
import onnxruntime as ort

csv_datei = "beton_mischungen_onnx.csv"
onnx_datei = "bestes_mlp_betonstaerke.onnx"
mischungen_df = pd.read_csv(csv_datei)
session = ort.InferenceSession(onnx_datei, providers=["CPUExecutionProvider"])
eingabespalten = json.loads(session.get_modelmeta().custom_metadata_map["feature_names"])

fehlende_spalten = [spalte for spalte in eingabespalten if spalte not in mischungen_df.columns]
if fehlende_spalten:
    raise ValueError(f"In der CSV fehlen Eingabespalten: {fehlende_spalten}")
if mischungen_df.empty:
    raise ValueError("Die CSV enthält keine Betonmischungen.")

# Nur Modellmerkmale verwenden; weitere CSV-Spalten bleiben in der Tabelle erhalten.
eingaben = mischungen_df.loc[:, eingabespalten].to_numpy(dtype=np.float32)
if not np.isfinite(eingaben).all():
    raise ValueError("Die Eingabedaten enthalten fehlende oder unendliche Werte.")

# Skalierung und Rückskalierung sind bereits Teil des ONNX-Modells.
praediktionen = session.run(None, {session.get_inputs()[0].name: eingaben})[0]
mischungen_df["PredStrength"] = praediktionen.ravel()
mischungen_df.to_csv("beton_mischungen_onnx_mit_praediktionen.csv", index=False)
mischungen_df


,CementComponent,BlastFurnaceSlag,FlyAshComponent,WaterComponent,SuperplasticizerComponent,CoarseAggregateComponent,FineAggregateComponent,AgeInDays,PredStrength
0,362.6,189.0,0.0,164.9,11.6,944.7,755.8,28,71.120537
1,190.0,190.0,0.0,228.0,0.0,932.0,670.0,90,43.293777
2,152.6,238.7,0.0,200.0,6.3,1001.8,683.9,28,26.016209


In [28]:
mischungen_df

,CementComponent,BlastFurnaceSlag,FlyAshComponent,WaterComponent,SuperplasticizerComponent,CoarseAggregateComponent,FineAggregateComponent,AgeInDays,PredStrength
0,362.6,189.0,0.0,164.9,11.6,944.7,755.8,28,71.120537
1,190.0,190.0,0.0,228.0,0.0,932.0,670.0,90,43.293777
2,152.6,238.7,0.0,200.0,6.3,1001.8,683.9,28,26.016209


In [29]:
session

In [30]:
eingabespalten

['CementComponent',
 'BlastFurnaceSlag',
 'FlyAshComponent',
 'WaterComponent',
 'SuperplasticizerComponent',
 'CoarseAggregateComponent',
 'FineAggregateComponent',
 'AgeInDays']

In [32]:
session.get_modelmeta().custom_metadata_map["feature_names"]

'["CementComponent", "BlastFurnaceSlag", "FlyAshComponent", "WaterComponent", "SuperplasticizerComponent", "CoarseAggregateComponent", "FineAggregateComponent", "AgeInDays"]'

In [33]:
fehlende_spalten

[]

In [34]:
eingaben

array([[ 362.6,  189. ,    0. ,  164.9,   11.6,  944.7,  755.8,   28. ],
       [ 190. ,  190. ,    0. ,  228. ,    0. ,  932. ,  670. ,   90. ],
       [ 152.6,  238.7,    0. ,  200. ,    6.3, 1001.8,  683.9,   28. ]],
      dtype=float32)

In [35]:
session.get_inputs()[0].name

'mischungen'

In [36]:
praediktionen

array([[71.12054 ],
       [43.293777],
       [26.016209]], dtype=float32)

In [37]:
praediktionen.ravel()

array([71.12054 , 43.293777, 26.016209], dtype=float32)